# IKSolver Inference Test

Tests `ik.inference.IKSolver` — the user-facing API.
No dataset loading required: only the model weights and MinMax files.

In [ ]:
import sys, json
from pathlib import Path

REPO_ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path().resolve().parent
sys.path.insert(0, str(REPO_ROOT))

import numpy as np
from ik.inference import IKSolver
from ik.kinematics.fk import FK
from ik.viperx300 import JOINT_LIMITS

solver = IKSolver()

## Test 1 — Single solve

Give one current joint config and one target transform, get back the solution.

In [ ]:
np.random.seed(0)

q_true  = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])
T_target = FK(q_true)   # ground-truth transform
q_init  = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])  # where robot currently is

result = solver.solve(q_init, T_target)

print(f"converged  : {result['converged']}")
print(f"iterations : {result['iterations']}")
print(f"q_mlp      : {np.round(result['q_mlp'], 4)}")
print(f"q_solution : {np.round(result['q'], 4)}")

# Task-space error of the final solution
T_sol = FK(result['q'])
pos_err = np.linalg.norm(T_target[:3, 3] - T_sol[:3, 3])
print(f"pos error  : {pos_err*100:.4f} cm")

## Test 2 — Batch solve

Solve N targets at once and summarise convergence and iteration counts.

In [ ]:
np.random.seed(1)
N = 200

q_trues   = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], (N, 6))
T_targets = np.array([FK(q) for q in q_trues])
q_inits   = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1], (N, 6))

results = solver.solve_batch(q_inits, T_targets)

iters     = np.array([r['iterations'] for r in results])
converged = np.array([r['converged']  for r in results])
pos_errs  = np.array([np.linalg.norm(T_targets[i, :3, 3] - FK(results[i]['q'])[:3, 3])
                       for i in range(N)])

print(f"N                  : {N}")
print(f"converged          : {converged.sum()} / {N}  ({100*converged.mean():.1f}%)")
print(f"mean iterations    : {iters.mean():.2f}")
print(f"mean pos error (cm): {pos_errs.mean()*100:.4f}")
print(f"max  pos error (cm): {pos_errs.max()*100:.4f}")

## Test 3 — Iteration distribution (batch)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(iters, bins=range(0, iters.max() + 2), color='steelblue', alpha=0.8, edgecolor='white')
axes[0].axvline(iters.mean(), color='black', linestyle='--', label=f'mean={iters.mean():.2f}')
axes[0].set_xlabel('NR iterations')
axes[0].set_title('Iteration distribution (MLP warm start)')
axes[0].legend()

axes[1].hist(pos_errs * 100, bins=40, color='mediumpurple', alpha=0.8)
axes[1].axvline(pos_errs.mean() * 100, color='black', linestyle='--',
                label=f'mean={pos_errs.mean()*100:.4f} cm')
axes[1].set_xlabel('Position error (cm)')
axes[1].set_title('Task-space position error after NR')
axes[1].legend()

plt.tight_layout()
plt.show()

## Test 4 — Edge cases

- All-zero start (home position)
- Target = current position (trivial solve)

In [ ]:
# Home start
q_home = np.zeros(6)
np.random.seed(2)
q_rand = np.random.uniform(JOINT_LIMITS[:, 0], JOINT_LIMITS[:, 1])
T_rand = FK(q_rand)

r_home = solver.solve(q_home, T_rand)
print("From home (zeros):")
print(f"  converged={r_home['converged']}  iters={r_home['iterations']}  "
      f"pos err={np.linalg.norm(FK(r_home['q'])[:3,3] - T_rand[:3,3])*100:.4f} cm")

# Trivial: target = current
r_trivial = solver.solve(q_rand, FK(q_rand))
print("\nTrivial (target = current):")
print(f"  converged={r_trivial['converged']}  iters={r_trivial['iterations']}  "
      f"pos err={np.linalg.norm(FK(r_trivial['q'])[:3,3] - FK(q_rand)[:3,3])*100:.4f} cm")